# Bài 25: API key linh hoạt
## Key mặc định của app + cho user nhập key riêng

**Mục tiêu:**
- Đọc key mặc định từ `.env` (local) hoặc `st.secrets` (cloud)
- Cho user nhập key riêng ở sidebar (tùy chọn) → ưu tiên key đó
- Chuyển `client` cố định → hàm `get_client()` phân giải key lúc chạy

**Vì sao cần:** để deploy công khai. Recruiter click link là chạy ngay (key của bạn), ai muốn dùng key riêng thì dán vào. Đây là bước chuẩn bị cho bài 26 (deploy).

---
## Phần 1: Lý thuyết

### Vì sao không dùng `client` cố định được nữa?

Hiện tại `config.py` làm:
```python
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))   # tạo 1 LẦN lúc import
```

Key bị 'đóng băng' lúc import. Nhưng giờ key có thể đến từ **3 nguồn, quyết định lúc chạy**:
1. Key user nhập ở sidebar (ưu tiên cao nhất)
2. `.env` (chạy local)
3. `st.secrets` (chạy trên Streamlit Cloud)

→ Phải thay bằng **hàm** `get_client()` phân giải key mỗi lần gọi.

---

### Thứ tự ưu tiên

```
get_client()
   ├─ 1. st.session_state["user_api_key"]   (user tự nhập) — nếu có
   └─ 2. _default_key():  os.getenv("GOOGLE_API_KEY")  (.env local)
                       →  st.secrets["GOOGLE_API_KEY"]  (cloud)
```

---

### `st.secrets` — key trên cloud

Trên Streamlit Cloud không có file `.env`. Thay vào đó bạn dán key vào phần **Secrets** của app (định dạng TOML):
```toml
GOOGLE_API_KEY = "AIza..."
```
Code đọc qua `st.secrets["GOOGLE_API_KEY"]`. (Local không có secrets thì `st.secrets` ném lỗi → phải bọc `try`.)

---

### Cache client theo key

`get_client()` bị gọi ở mỗi lần LLM chạy (rút mã, phân tích, detect). Tạo `genai.Client` mới mỗi lần thì phí. Cache theo key bằng `lru_cache`:
```python
@lru_cache(maxsize=4)
def _make_client(api_key: str):
    return genai.Client(api_key=api_key)
```
Cùng 1 key → trả lại client cũ. Đổi key → tạo client mới (và cache tiếp).

---
## Phần 2: Ví dụ — logic phân giải key

Mô phỏng `_default_key()` (không cần Streamlit). Chạy thử với/không có biến môi trường.

In [1]:
import os

def default_key_demo() -> str | None:
    # 1. thử .env / biến môi trường
    key = os.getenv("GOOGLE_API_KEY")
    if key:
        return f"(từ env) ...{key[-4:]}"
    # 2. thử st.secrets — bọc try vì local không có
    try:
        import streamlit as st
        if "GOOGLE_API_KEY" in st.secrets:
            return "(từ st.secrets)"
    except Exception:
        pass
    return None

from dotenv import load_dotenv
load_dotenv()
print("Key mặc định phân giải được:", default_key_demo())

Key mặc định phân giải được: (từ env) ...T0wg


Chạy local có `.env` → thấy `(từ env) ...xxxx`. Trên cloud (không env) sẽ rơi xuống `st.secrets`.

---
## Phần 3: Bài tập

### Bước 1: Sửa `app/config.py`

Bỏ dòng `client = genai.Client(...)` cố định. Thay bằng 3 hàm (điền `TODO`):

```python
import os
from functools import lru_cache
from dotenv import load_dotenv
from google import genai

load_dotenv()

MODEL_NAME = "gemini-2.5-flash"
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
CHROMA_PATH = os.path.join(PROJECT_ROOT, "data", "chroma_db")
COLLECTION_NAME = "multi_company_reports"


def _default_key() -> str | None:
    """Key mặc định: .env (local) hoặc st.secrets (cloud)."""
    key = os.getenv("GOOGLE_API_KEY")
    if key:
        return key
    try:
        import streamlit as st
        if "GOOGLE_API_KEY" in st.secrets:
            return st.secrets["GOOGLE_API_KEY"]
    except Exception:
        pass
    return None


@lru_cache(maxsize=4)
def _make_client(api_key: str):
    return genai.Client(api_key=api_key)


def get_client():
    """Gemini client. Ưu tiên key user nhập, không thì dùng key mặc định."""
    api_key = None
    try:
        import streamlit as st
        api_key = st.session_state.get("user_api_key")   # key user nhập ở sidebar
    except Exception:
        pass
    # TODO: nếu api_key rỗng/None → dùng _default_key()
    if not api_key:
        api_key = _default_key()
    return _make_client(api_key)
```

---

### Bước 2: Đổi `client` → `get_client()` ở 3 nơi dùng LLM

Các file đang `from config import client` rồi gọi `client.models.generate_content(...)`. Đổi thành import `get_client` và gọi `get_client().models.generate_content(...)`:

| File | Sửa |
|------|-----|
| `app/agents/analysis.py` | `from config import get_client, MODEL_NAME` → `get_client().models.generate_content(...)` |
| `app/agents/symbol_extraction.py` | tương tự |
| `app/services/ingest.py` | tương tự (trong `detect_symbol_from_pdf`) |

> Ví dụ ở `analysis.py`:
> ```python
> from config import get_client, MODEL_NAME       # thay 'client' bằng 'get_client'
> ...
> response = get_client().models.generate_content(model=MODEL_NAME, contents=prompt)
> ```

---

### Bước 3: Ô nhập key riêng ở sidebar — `app/main.py`

Thêm ngay dưới `st.header("Cài đặt")`:

```python
    with st.expander("🔑 Dùng API key riêng (tùy chọn)"):
        st.text_input(
            "Gemini API key của bạn",
            type="password",
            key="user_api_key",      # ← đúng key mà get_client() đọc
            help="Để trống = dùng key mặc định của app.",
        )
```

> Mẹo hay: đặt `key="user_api_key"` cho ô nhập → Streamlit tự lưu giá trị vào `st.session_state["user_api_key"]`, đúng chỗ `get_client()` đọc. Không cần code copy qua lại.

---

### Bước 4: Test

1. `streamlit run app/main.py` — chạy như cũ (dùng key `.env`), hỏi đáp bình thường → xác nhận `get_client()` hoạt động với key mặc định
2. Mở '🔑 Dùng API key riêng', dán một key Gemini hợp lệ khác → hỏi 1 câu → vẫn chạy (giờ dùng key đó)
3. Dán key sai → hỏi → thấy lỗi API key invalid (đúng, vì đang dùng key bạn nhập)
4. Xóa ô key → quay lại dùng key mặc định

**Checklist:**
- [ ] Không còn `client` cố định trong `config.py`; mọi nơi dùng `get_client()`
- [ ] Không nhập gì → chạy bằng key mặc định (.env)
- [ ] Nhập key riêng → app dùng key đó